<a href="https://colab.research.google.com/github/yogeshwardev/csa6301_Thread_intelligence_network_security/blob/main/lap%20program%20outputs%20colab%20from%2014_to_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def normalize_ioc(raw):
    return {
        "type": raw.get("type", "").lower(),
        "value": raw.get("value", "").strip().lower(),
        "source": raw.get("source", "unknown"),
        "severity": raw.get("severity", "medium").lower(),
    }


raw_feed = [
    {"type": "IP", "value": " 45.33.32.156 ", "source": "MISP", "severity": "High"},
    {"type": "Domain", "value": "Malicious-Site.COM", "source": "VirusTotal", "severity": "critical"},
    {"type": "Hash", "value": "5d41402abc4b2a76b9719d911017c592", "source": "AlienVault OTX", "severity": "Medium"},
]

tip_database = [normalize_ioc(r) for r in raw_feed]

internal_firewall_logs = [
    {"src_ip": "45.33.32.156", "dest": "internal-server-1", "action": "connection attempt"},
    {"src_ip": "10.0.0.5", "dest": "internal-server-2", "action": "connection attempt"},
]


def correlate_with_tip(logs, tip_db):
    ioc_ips = {i["value"] for i in tip_db if i["type"] == "ip"}
    return [
        {"alert": "Known malicious IP contacted internal system", "log": log}
        for log in logs
        if log["src_ip"] in ioc_ips
    ]


alerts = correlate_with_tip(internal_firewall_logs, tip_database)

print(f"Normalized {len(tip_database)} IOCs:")
for ioc in tip_database:
    print(" ", ioc)
print(f"Generated {len(alerts)} alert(s):", alerts)


# Test Cases
def test_experiment14():
    assert tip_database[0]["value"] == "45.33.32.156", "IOC values should be trimmed and lowercased"
    assert tip_database[1]["value"] == "malicious-site.com"
    assert tip_database[0]["type"] == "ip"
    assert len(alerts) == 1, "Exactly one firewall log should match a known malicious IP"
    assert alerts[0]["log"]["src_ip"] == "45.33.32.156"
    print("Experiment 14: All test cases passed.")


test_experiment14()


Normalized 3 IOCs:
  {'type': 'ip', 'value': '45.33.32.156', 'source': 'MISP', 'severity': 'high'}
  {'type': 'domain', 'value': 'malicious-site.com', 'source': 'VirusTotal', 'severity': 'critical'}
  {'type': 'hash', 'value': '5d41402abc4b2a76b9719d911017c592', 'source': 'AlienVault OTX', 'severity': 'medium'}
Generated 1 alert(s): [{'alert': 'Known malicious IP contacted internal system', 'log': {'src_ip': '45.33.32.156', 'dest': 'internal-server-1', 'action': 'connection attempt'}}]
Experiment 14: All test cases passed.


In [2]:
def correlate_logs(firewall_log, windows_log, vpn_log):
    indicators = 0
    reasons = []
    if firewall_log.get("event") == "failed_login":
        indicators += 1
        reasons.append("Failed login at firewall")
    if windows_log.get("event") == "account_locked":
        indicators += 1
        reasons.append("Windows account locked")
    if vpn_log.get("failed_attempts", 0) >= 3:
        indicators += 1
        reasons.append("Repeated VPN authentication failures")
    return {
        "indicators": indicators,
        "reasons": reasons,
        "brute_force_detected": indicators >= 2,
    }


scenario_attack = correlate_logs(
    {"event": "failed_login", "ip": "192.168.10.5"},
    {"event": "account_locked", "user": "jdoe"},
    {"failed_attempts": 5},
)

scenario_normal = correlate_logs(
    {"event": "login_success"},
    {"event": "none"},
    {"failed_attempts": 0},
)

print("Attack scenario:", scenario_attack)
print("Normal scenario:", scenario_normal)


# Test Cases
def test_experiment15():
    assert scenario_attack["brute_force_detected"] is True
    assert scenario_attack["indicators"] == 3
    assert scenario_normal["brute_force_detected"] is False
    assert scenario_normal["indicators"] == 0
    print("Experiment 15: All test cases passed.")


test_experiment15()


Attack scenario: {'indicators': 3, 'reasons': ['Failed login at firewall', 'Windows account locked', 'Repeated VPN authentication failures'], 'brute_force_detected': True}
Normal scenario: {'indicators': 0, 'reasons': [], 'brute_force_detected': False}
Experiment 15: All test cases passed.


In [3]:
import statistics

# Hypothesis-based: "attackers may use encoded PowerShell commands"
execution_logs = [
    {"process": "powershell.exe", "args": "-enc SGVsbG8=", "user": "svc_admin"},
    {"process": "notepad.exe", "args": "", "user": "jdoe"},
    {"process": "powershell.exe", "args": "-nop -w hidden -enc ZXZpbA==", "user": "jdoe"},
]


def hypothesis_hunt_powershell(logs):
    return [l for l in logs if l["process"] == "powershell.exe" and "-enc" in l["args"]]


# Intelligence-based: match connections against a known-malicious-IP feed
connection_logs = [{"ip": "45.33.32.156"}, {"ip": "8.8.8.8"}, {"ip": "185.220.101.1"}]
known_malicious_ips = {"45.33.32.156", "185.220.101.1"}


def intelligence_hunt(logs, feed):
    return [l for l in logs if l["ip"] in feed]


# Analytics-based: flag a login hour that deviates far from the user's normal pattern
login_hours_history = [9, 9, 10, 9, 8, 9, 10]
new_login_hour = 3


def analytics_hunt(history, new_value, threshold=2.0):
    mean = statistics.mean(history)
    stdev = statistics.pstdev(history) or 1
    z = abs(new_value - mean) / stdev
    return {"z_score": round(z, 2), "anomalous": z > threshold}


hyp_result = hypothesis_hunt_powershell(execution_logs)
intel_result = intelligence_hunt(connection_logs, known_malicious_ips)
analytics_result = analytics_hunt(login_hours_history, new_login_hour)

print("Hypothesis-based hits:", hyp_result)
print("Intelligence-based hits:", intel_result)
print("Analytics-based result:", analytics_result)


# Test Cases
def test_experiment16():
    assert len(hyp_result) == 2, "Both encoded PowerShell executions should be flagged"
    assert all(r["process"] == "powershell.exe" for r in hyp_result)
    assert len(intel_result) == 2, "Both known-malicious IPs should be matched"
    assert analytics_result["anomalous"] is True, "A 3 AM login should be anomalous against an 8-10 AM history"
    print("Experiment 16: All test cases passed.")


test_experiment16()


Hypothesis-based hits: [{'process': 'powershell.exe', 'args': '-enc SGVsbG8=', 'user': 'svc_admin'}, {'process': 'powershell.exe', 'args': '-nop -w hidden -enc ZXZpbA==', 'user': 'jdoe'}]
Intelligence-based hits: [{'ip': '45.33.32.156'}, {'ip': '185.220.101.1'}]
Analytics-based result: {'z_score': 9.62, 'anomalous': True}
Experiment 16: All test cases passed.


In [4]:
raw_events = [
    {"source": "firewall", "ip": "45.33.32.156", "event": "blocked"},
    {"source": "antivirus", "host": "WIN10-05", "event": "malware_detected", "file": "invoice.exe"},
    {"source": "server", "event": "login_success", "user": "admin"},
]


def normalize(events):
    return [
        {
            "source": e["source"],
            "event_type": e["event"],
            "severity": "high" if e["event"] in ("malware_detected", "blocked") else "low",
        }
        for e in events
    ]


def correlation_engine(normalized_events):
    return [e for e in normalized_events if e["severity"] == "high"]


def generate_alerts(high_severity_events):
    return [f"ALERT: {e['source']} reported {e['event_type']}" for e in high_severity_events]


normalized = normalize(raw_events)
correlated = correlation_engine(normalized)
siem_alerts = generate_alerts(correlated)

print("Normalized:", normalized)
print("Correlated (high severity):", correlated)
print("Alerts:", siem_alerts)


# Test Cases
def test_experiment17():
    assert len(normalized) == 3
    assert len(correlated) == 2, "Only the 2 high-severity events should reach correlation"
    assert len(siem_alerts) == 2
    assert all(a.startswith("ALERT:") for a in siem_alerts)
    print("Experiment 17: All test cases passed.")


test_experiment17()


Normalized: [{'source': 'firewall', 'event_type': 'blocked', 'severity': 'high'}, {'source': 'antivirus', 'event_type': 'malware_detected', 'severity': 'high'}, {'source': 'server', 'event_type': 'login_success', 'severity': 'low'}]
Correlated (high severity): [{'source': 'firewall', 'event_type': 'blocked', 'severity': 'high'}, {'source': 'antivirus', 'event_type': 'malware_detected', 'severity': 'high'}]
Alerts: ['ALERT: firewall reported blocked', 'ALERT: antivirus reported malware_detected']
Experiment 17: All test cases passed.


In [5]:
auth_log = [
    ("09:15", "login_success"),
    ("09:17", "failed_login"),
    ("09:18", "failed_login"),
    ("09:19", "failed_login"),
    ("09:20", "account_locked"),
]


def detect_bruteforce(log, fail_threshold=3):
    consecutive_fails = 0
    for _, event in log:
        if event == "failed_login":
            consecutive_fails += 1
        elif event == "account_locked" and consecutive_fails >= fail_threshold:
            return True, consecutive_fails
        else:
            consecutive_fails = 0
    return False, consecutive_fails


def classify_log_line(line):
    keywords = {
        "firewall": ["blocked", "dropped", "denied"],
        "security": ["login", "locked", "authentication"],
        "web_server": ["get", "post", "http"],
    }
    line_lower = line.lower()
    for category, words in keywords.items():
        if any(w in line_lower for w in words):
            return category
    return "unknown"


detected, fails = detect_bruteforce(auth_log)
sample_classification = classify_log_line("User login failed - authentication error")

print("Brute-force detected:", detected, "| consecutive fails before lockout:", fails)
print("Log line classified as:", sample_classification)


# Test Cases
def test_experiment18():
    assert detected is True
    assert fails == 3
    normal_log = [("10:00", "login_success"), ("10:05", "login_success")]
    assert detect_bruteforce(normal_log)[0] is False
    assert sample_classification == "security"
    assert classify_log_line("Connection blocked by firewall rule 12") == "firewall"
    print("Experiment 18: All test cases passed.")


test_experiment18()


Brute-force detected: True | consecutive fails before lockout: 3
Log line classified as: security
Experiment 18: All test cases passed.


In [6]:
import random

random.seed(42)


def generate_event_stream(n=20):
    stream = []
    for _ in range(n):
        stream.append({
            "hour": random.choice([9, 10, 11, 14, 15, 16]),  # normal business hours
            "failed_logins": random.choice([0, 0, 0, 1]),
        })
    # inject 2 known anomalies
    stream.append({"hour": 3, "failed_logins": 0})
    stream.append({"hour": 10, "failed_logins": 9})
    return stream


def monitor_stream(stream):
    flags = []
    for event in stream:
        reasons = []
        if event["hour"] < 6 or event["hour"] > 22:
            reasons.append("off-hours activity")
        if event["failed_logins"] >= 5:
            reasons.append("excessive failed logins")
        if reasons:
            flags.append({"event": event, "reasons": reasons})
    return flags


event_stream = generate_event_stream()
flags = monitor_stream(event_stream)

print(f"Stream length: {len(event_stream)} | Flags raised: {len(flags)}")
for f in flags:
    print(f)


# Test Cases
def test_experiment19():
    assert len(flags) == 2, "Exactly the 2 injected anomalies should be flagged"
    assert any("off-hours activity" in f["reasons"] for f in flags)
    assert any("excessive failed logins" in f["reasons"] for f in flags)
    print("Experiment 19: All test cases passed.")


test_experiment19()


Stream length: 22 | Flags raised: 2
{'event': {'hour': 3, 'failed_logins': 0}, 'reasons': ['off-hours activity']}
{'event': {'hour': 10, 'failed_logins': 9}, 'reasons': ['excessive failed logins']}
Experiment 19: All test cases passed.


In [7]:
tool_catalog = {
    "Wireshark": {
        "purpose": "packet analysis",
        "keywords": ["packet analysis", "packet capture", "network traffic"],
    },
    "Nmap": {
        "purpose": "network scanning",
        "keywords": ["network scanning", "open ports", "port scan"],
    },
    "Nessus": {
        "purpose": "vulnerability assessment",
        "keywords": ["vulnerability assessment", "known vulnerabilities"],
    },
    "Burp Suite": {
        "purpose": "web application testing",
        "keywords": ["web application testing", "http proxy"],
    },
    "Metasploit": {
        "purpose": "penetration testing",
        "keywords": ["penetration testing", "exploit code"],
    },
    "Splunk": {
        "purpose": "log management and SIEM",
        "keywords": ["log search", "siem dashboard"],
    },
    "ELK Stack": {
        "purpose": "log collection and visualization",
        "keywords": ["log visualization", "elasticsearch"],
    },
    "Snort": {
        "purpose": "intrusion detection",
        "keywords": ["intrusion detection"],
    },
    "VirusTotal": {
        "purpose": "malware scanning",
        "keywords": ["malware scanning", "file hash reputation"],
    },
    "MISP": {
        "purpose": "threat intelligence sharing",
        "keywords": ["threat intelligence sharing", "indicators of compromise"],
    },
}


def recommend_tool(task_description):
    task_lower = task_description.lower()
    scores = {
        tool: sum(1 for kw in info["keywords"] if kw in task_lower)
        for tool, info in tool_catalog.items()
    }
    scores = {t: s for t, s in scores.items() if s > 0}
    return max(scores, key=scores.get) if scores else None


test_tasks = [
    "We need to perform network scanning to find open ports on the target.",
    "The analyst wants to inspect packet capture data during the incident.",
    "Check the file hash reputation to see if it's known malware.",
    "Our team wants to enable threat intelligence sharing of indicators of compromise with partners.",
]

recommendations = [recommend_tool(t) for t in test_tasks]
print(list(zip(test_tasks, recommendations)))


# Test Cases
def test_experiment20():
    assert recommendations[0] == "Nmap"
    assert recommendations[1] == "Wireshark"
    assert recommendations[2] == "VirusTotal"
    assert recommendations[3] == "MISP"
    assert recommend_tool("completely unrelated text about cooking pasta") is None
    print("Experiment 20: All test cases passed.")


test_experiment20()


[('We need to perform network scanning to find open ports on the target.', 'Nmap'), ('The analyst wants to inspect packet capture data during the incident.', 'Wireshark'), ("Check the file hash reputation to see if it's known malware.", 'VirusTotal'), ('Our team wants to enable threat intelligence sharing of indicators of compromise with partners.', 'MISP')]
Experiment 20: All test cases passed.


In [8]:
splunk_index = [
    {"_time": "09:15", "source": "auth", "user": "jdoe", "action": "login_success"},
    {"_time": "09:17", "source": "auth", "user": "jdoe", "action": "login_failed"},
    {"_time": "09:18", "source": "auth", "user": "jdoe", "action": "login_failed"},
    {"_time": "09:20", "source": "firewall", "ip": "45.33.32.156", "action": "blocked"},
    {"_time": "09:21", "source": "auth", "user": "asmith", "action": "login_success"},
]


def spl_search(index, **filters):
    results = index
    for field, value in filters.items():
        results = [r for r in results if r.get(field) == value]
    return results


def spl_stats_count_by(index, field):
    counts = {}
    for r in index:
        key = r.get(field, "unknown")
        counts[key] = counts.get(key, 0) + 1
    return counts


failed_logins = spl_search(splunk_index, source="auth", action="login_failed")
counts_by_source = spl_stats_count_by(splunk_index, "source")

print("search source=auth action=login_failed ->", failed_logins)
print("stats count by source ->", counts_by_source)


# Test Cases
def test_experiment21():
    assert len(failed_logins) == 2
    assert all(r["user"] == "jdoe" for r in failed_logins)
    assert counts_by_source == {"auth": 4, "firewall": 1}
    print("Experiment 21: All test cases passed.")


test_experiment21()


search source=auth action=login_failed -> [{'_time': '09:17', 'source': 'auth', 'user': 'jdoe', 'action': 'login_failed'}, {'_time': '09:18', 'source': 'auth', 'user': 'jdoe', 'action': 'login_failed'}]
stats count by source -> {'auth': 4, 'firewall': 1}
Experiment 21: All test cases passed.


In [9]:
import re

raw_log_lines = [
    "2026-07-25 09:15:01 INFO auth: user=jdoe action=login_success ip=10.0.0.5",
    "2026-07-25 09:17:32 WARN auth: user=jdoe action=login_failed ip=10.0.0.5",
    "2026-07-25 09:20:10 ERROR firewall: action=blocked ip=45.33.32.156",
]


def logstash_parse(line):
    pattern = r"(?P<date>\S+) (?P<time>\S+) (?P<level>\w+) (?P<source>\w+): (?P<fields>.*)"
    m = re.match(pattern, line)
    if not m:
        return None
    doc = m.groupdict()
    field_str = doc.pop("fields")
    for kv in field_str.split():
        if "=" in kv:
            k, v = kv.split("=", 1)
            doc[k] = v
    return doc


elasticsearch_index = [logstash_parse(l) for l in raw_log_lines]


def es_search(index, **query):
    return [doc for doc in index if all(doc.get(k) == v for k, v in query.items())]


def kibana_aggregate(index, field):
    agg = {}
    for doc in index:
        key = doc.get(field, "unknown")
        agg[key] = agg.get(key, 0) + 1
    return agg


parsed_ok = all(d is not None for d in elasticsearch_index)
failed_search = es_search(elasticsearch_index, action="login_failed")
level_aggregation = kibana_aggregate(elasticsearch_index, "level")

print("Parsed documents:")
for d in elasticsearch_index:
    print(" ", d)
print("Search action=login_failed ->", failed_search)
print("Kibana aggregation by level ->", level_aggregation)


# Test Cases
def test_experiment22():
    assert parsed_ok, "All raw log lines should be parsed by the Logstash-style parser"
    assert len(failed_search) == 1
    assert failed_search[0]["user"] == "jdoe"
    assert level_aggregation == {"INFO": 1, "WARN": 1, "ERROR": 1}
    print("Experiment 22: All test cases passed.")


test_experiment22()


Parsed documents:
  {'date': '2026-07-25', 'time': '09:15:01', 'level': 'INFO', 'source': 'auth', 'user': 'jdoe', 'action': 'login_success', 'ip': '10.0.0.5'}
  {'date': '2026-07-25', 'time': '09:17:32', 'level': 'WARN', 'source': 'auth', 'user': 'jdoe', 'action': 'login_failed', 'ip': '10.0.0.5'}
  {'date': '2026-07-25', 'time': '09:20:10', 'level': 'ERROR', 'source': 'firewall', 'action': 'blocked', 'ip': '45.33.32.156'}
Search action=login_failed -> [{'date': '2026-07-25', 'time': '09:17:32', 'level': 'WARN', 'source': 'auth', 'user': 'jdoe', 'action': 'login_failed', 'ip': '10.0.0.5'}]
Kibana aggregation by level -> {'INFO': 1, 'WARN': 1, 'ERROR': 1}
Experiment 22: All test cases passed.


In [10]:
import hashlib

sample_file_content = (
    b"MZ\x90\x00...fake_pe_header...CreateRemoteThread...WriteProcessMemory..."
)
suspicious_api_keywords = [
    b"CreateRemoteThread",
    b"WriteProcessMemory",
    b"RegSetValue",
    b"InternetOpenUrl",
]


def static_analysis(file_bytes):
    file_hash = hashlib.sha256(file_bytes).hexdigest()
    found_apis = [
        api.decode() for api in suspicious_api_keywords if api in file_bytes
    ]
    return {
        "sha256": file_hash,
        "suspicious_apis_found": found_apis,
        "static_risk_score": len(found_apis),
    }


def dynamic_analysis(sandbox_log):
    indicators = 0
    reasons = []
    if sandbox_log.get("registry_writes", 0) > 0:
        indicators += 1
        reasons.append("Modified registry")
    if sandbox_log.get("network_connections", 0) > 0:
        indicators += 1
        reasons.append("Made outbound network connection")
    if sandbox_log.get("new_processes_spawned", 0) > 1:
        indicators += 1
        reasons.append("Spawned multiple child processes")
    return {"dynamic_risk_score": indicators, "reasons": reasons}


def classify(static_result, dynamic_result, threshold=2):
    total = static_result["static_risk_score"] + dynamic_result["dynamic_risk_score"]
    return "malicious" if total >= threshold else "benign"


static_result = static_analysis(sample_file_content)
sandbox_log = {
    "registry_writes": 3,
    "network_connections": 1,
    "new_processes_spawned": 2,
}
dynamic_result = dynamic_analysis(sandbox_log)
verdict = classify(static_result, dynamic_result)

benign_static = static_analysis(
    b"just a normal text file with no suspicious content"
)
benign_dynamic = dynamic_analysis(
    {"registry_writes": 0, "network_connections": 0, "new_processes_spawned": 0}
)
benign_verdict = classify(benign_static, benign_dynamic)

print("Static analysis:", static_result)
print("Dynamic analysis:", dynamic_result)
print("Verdict:", verdict)
print("Benign file verdict:", benign_verdict)


# Test Cases
def test_experiment23():
    assert len(static_result["sha256"]) == 64
    assert "CreateRemoteThread" in static_result["suspicious_apis_found"]
    assert static_result["static_risk_score"] >= 2
    assert dynamic_result["dynamic_risk_score"] == 3
    assert verdict == "malicious"
    assert benign_verdict == "benign"
    print("Experiment 23: All test cases passed.")


test_experiment23()


Static analysis: {'sha256': '25329da60625b452686ec727bb3d0f706b638afbd6fcfda5720437ab56030222', 'suspicious_apis_found': ['CreateRemoteThread', 'WriteProcessMemory'], 'static_risk_score': 2}
Dynamic analysis: {'dynamic_risk_score': 3, 'reasons': ['Modified registry', 'Made outbound network connection', 'Spawned multiple child processes']}
Verdict: malicious
Benign file verdict: benign
Experiment 23: All test cases passed.
